<a href="https://colab.research.google.com/github/davejeon/dsr_agentic_workflow/blob/main/Exercise_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install packages
%pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.8 MB/s eta 0:00:00


In [2]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from google.colab import userdata

In [3]:
# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT')
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
# Test the output
userdata.get('LANGSMITH_PROJECT')

'dsr_agentic_ai'

We will start exploring the use of AI Agents using the core components of messages.

Messages are the fundamental unit of context for models in LangChain.

Here the unit is an object that contains information about:

- The role (whether system or user)
- Content (actual contents of the message)
- Metadata (response information)

They represent the input, metadata, and output of the conversation when interacting with an llm.

In [21]:
# Initialise the LLM Model
model = init_chat_model("google_genai:gemini-3.1-flash-lite")

Message content is a data payload that gets sent to and from the model with a number of attributes.

The input into the model, is a **HumanMessage** object, which represents the user's input.


The output of the model, is a **AIMessage** objecttype, which represents the model's output.

A **SystemMessage** represents an initial set of instructiontions that primes the model's behaviour. Use it to set the tone, define the model's role, and establish guidelines for responses.

In [22]:
system_msg = SystemMessage("""You are a medieval poet and you only answer
in the form of limericks.""")

messages = [
    system_msg,
    HumanMessage("What is the capital of France?")
]


In [23]:
# Interact with the LLM Model
response = model.invoke(messages)
print(response)

content=[{'type': 'text', 'text': 'In a kingdom of wine and romance,\nLies the heart of the nation of France.\nIn Paris they dwell,\nWhere the fountains all swell,\nAnd the nobles in velvet will dance.', 'extras': {'signature': 'EjQKMgEMOdbHtwGfpPs77TF1AT/HESNEXQCzD2D1pxsaeLbRdKTnnzzVwkxWUXnxSjxiR+R5'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e268f-5d50-7ca0-b72a-ef36fdfe2704-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 41, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}


In [25]:
print(f"Response contents: {response.content_blocks}")
print(f"Data type of the response conents: {type(response.content_blocks)}")
print("")
print(f"Data type of the first layer of the response contents: {type(response.content_blocks[0])}")
print("")
print(response.content_blocks[0]["text"])

Response contents: [{'type': 'text', 'text': 'In a kingdom of wine and romance,\nLies the heart of the nation of France.\nIn Paris they dwell,\nWhere the fountains all swell,\nAnd the nobles in velvet will dance.', 'extras': {'signature': 'EjQKMgEMOdbHtwGfpPs77TF1AT/HESNEXQCzD2D1pxsaeLbRdKTnnzzVwkxWUXnxSjxiR+R5'}}]
Data type of the response conents: <class 'list'>

Data type of the first layer of the response contents: <class 'dict'>

In a kingdom of wine and romance,
Lies the heart of the nation of France.
In Paris they dwell,
Where the fountains all swell,
And the nobles in velvet will dance.


In [26]:
# More metadata fun
print(response.usage_metadata)

{'input_tokens': 27, 'output_tokens': 41, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}


### **Parameters, parameters, and more darn parameters**

We can set the behaviour of our model at initialisation.

Some parameters  of the **init_chat_model** function we will look at are:

**temperature:** Between 0-2.This controls the randomness of the model output. The lower the number more deterministic.

**max_tokens:** Limits the number of tokens in the response. Mo' Tokens Mo' Money

**timeout:** The maximum time (in seconds) to wait for a response.

**max_retries**: Maximum time (in seconds) to wait for a response from the model before canceling request.

**Exercise:** Experiment with some of the parameters to control the output of the model.

Get really creative!

In [28]:
# Initialise the LLM Model
model = init_chat_model(model = "google_genai:gemini-3.1-flash-lite",
                        temperature = 2,
                        max_tokens = 200)
system_msg = SystemMessage("""You are a caveman and only respond in short terse expressions.""")

messages = [
    system_msg,
    HumanMessage("What is the cause of economic inequality?")
]

# Interact with the LLM Model
response = model.invoke(messages)
print(response.content_blocks[0]["text"])

Strong man take much. Weak man get none. Cave empty.
